In [ ]:
from __future__ import annotations

import json
import os
from collections import defaultdict
from typing import Any

import chromadb
from tqdm import auto as tqdm

from tklearn.agents.core import Tool, ToolCallingAgent
from tklearn.agents.core.tools import tool
from tklearn.agents.models import model_factory
from tklearn.embeddings import AutoEmbedding
from tklearn.kb import KnowledgeBase
from tklearn.nn.utils.devices import get_device


In [ ]:
kb = KnowledgeBase("wiktionary")

In [ ]:
embedder = AutoEmbedding({
    "loader": "transformers",
    "name": "sentence-transformers/all-mpnet-base-v2",
})

In [ ]:
client = chromadb.Client()

In [ ]:
if "wiktionary" in [col.name for col in client.list_collections()]:
    client.delete_collection("wiktionary")

collection = client.create_collection("wiktionary")

In [ ]:
sense_words = defaultdict(set)

for word, word_sense_idxes in kb.senses.items():
    for sense_idx in word_sense_idxes:
        sense_words[sense_idx].add(word)

In [ ]:
BATCH_SIZE = 5_000
DESC = "Adding to ChromaDB in batches"

sense_idxs = list(kb.idx2gloss.keys())

for i in tqdm.tqdm(range(0, len(sense_idxs), BATCH_SIZE), desc=DESC):
    batch_sense_idxs = sense_idxs[i : i + BATCH_SIZE]
    ids = [f"wiki:{idx}" for idx in batch_sense_idxs]
    documents = [kb.idx2gloss[idx] for idx in batch_sense_idxs]
    metadata = []
    embeddings = []
    for sense_idx in batch_sense_idxs:
        embeddings.append(kb.embeddings[sense_idx].tolist())
        metadata.append({
            "words": json.dumps(list(sense_words[sense_idx])),
            "sense_idx": sense_idx,
        })
    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadata,
        embeddings=embeddings,
    )

collection.query(
    query_embeddings=[embedder.encode("A domesticated carnivorous mammal.")],
    n_results=5,
)

In [ ]:
@tool
def wikitionary_definition_search(definition: str) -> Any:
    """Find word senses associated with a given definition from Wiktionary.

    Notes:
        For the definition argument, provide a string that describes the meaning of a word.

    Args:
        definition (str): The definition string to search for associated words.

    Returns:
        str: A formatted string containing the top words found.
    """
    results = collection.query(
        query_embeddings=[embedder.encode(definition)],
        n_results=5,
        # where_documents={"$contains": base_form},
    )
    words_list = []
    for doc, metadata in zip(results["documents"][0], results["metadatas"][0]):
        words = json.loads(metadata["words"])
        sense_idx = metadata["sense_idx"]
        words_list.append(
            f"Search Definition: {definition}\n"
            f"Words: {', '.join(words)}\nDefinition: {doc}\nSense Index: {sense_idx}\n"
        )
    return "\n".join(words_list)


@tool
def wikitionary_extract(text: str) -> Any:
    """Extract important words from a given text using Wiktionary.

    Notes:
        For the text argument, provide a string of text from which to extract important words.

    Args:
        text (str): The text string to extract important words from.

    Returns:
        str: A formatted string containing the top words & senses found along with their definitions.
    """
    outputs = []
    for mention in kb.extract_mentions(text):
        for candidate in mention.candidates:
            sense_id = candidate.sense_id
            definition = candidate.definition
            words = sense_words.get(sense_id, set())
            outputs.append(
                f"Mention: {mention.form}\n"
                f"Words: {', '.join(words)}\nDefinition: {definition}\nSense Index: {sense_id}\n"
            )
    return "\n".join(outputs)